# Crime Data

***
**This file will import and aggregate the crime data to create a large table of all crimes done.**  
**It will further use the supportive tables to add more context to the data.**
***

## Aggregate Data

This file will create a final database that can be analysed in visualisation software such as Tableau or PowerBI.

In [1]:
# Import modules
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

### New Crime ID

Firstly, This file will define a function to create a new Crime ID for the crime data.  
It should identify the police force, the year and month, and be able to store up to 5 digits worth of data (100,000 entries per month).  
It will start by using a sample piece of data.

#### Ingestion

In [2]:
## Import one months worth of data
police_region = 'south-yorkshire'
year_month = '2026-03' # Get the most recent data available

raw_sth_yk = pd.read_csv(f'../Data/Raw/crime-data/{police_region}/{year_month}-{police_region}-street.csv')

raw_sth_yk.head()

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
0,NaN,2026-03,South Yorkshire Police,South Yorkshire Police,-1.397019,53.054017,On or near Supermarket,E01019456,Amber Valley 005E,Anti-social behaviour,NaN,NaN
1,NaN,2026-03,South Yorkshire Police,South Yorkshire Police,-1.451708,53.599208,On or near Jack Close Orchard,E01007434,Barnsley 001A,Anti-social behaviour,NaN,NaN
2,NaN,2026-03,South Yorkshire Police,South Yorkshire Police,-1.454483,53.598502,On or near B6428,E01007434,Barnsley 001A,Anti-social behaviour,NaN,NaN
3,NaN,2026-03,South Yorkshire Police,South Yorkshire Police,-1.446274,53.603435,On or near Warren Close,E01007434,Barnsley 001A,Anti-social behaviour,NaN,NaN
4,NaN,2026-03,South Yorkshire Police,South Yorkshire Police,-1.450879,53.602548,On or near Ruston Drive,E01007434,Barnsley 001A,Anti-social behaviour,NaN,NaN


In [3]:
# Import the location lookup table
location_lookup = pd.read_csv('../Data/Processed/location-lookup-table.csv')

location_lookup.head()

,lsoa_code,lsoa_name,lad_code,lad_name,pfa_code,pfa_name
0,E01012000,Hartlepool 007E,E06000001,Hartlepool,E23000013,Cleveland
1,E01011964,Hartlepool 007B,E06000001,Hartlepool,E23000013,Cleveland
2,E01011999,Hartlepool 007D,E06000001,Hartlepool,E23000013,Cleveland
3,E01011967,Hartlepool 007C,E06000001,Hartlepool,E23000013,Cleveland
4,E01011951,Hartlepool 007A,E06000001,Hartlepool,E23000013,Cleveland


#### Cleaning and Validation

Using the cleaning function described in file 01:

In [4]:
def Clean_Crime(raw_crime, dropped_rows):
    # Drop useless columns
    crime = raw_crime[['Crime ID', 'LSOA code', 'Month', 'Latitude', 'Longitude', 'Crime type', 'Falls within']]

    # Rename columns
    crime = crime.rename(columns={
        'Crime ID': 'old_crime_id',
        'LSOA code': 'lsoa_code',
        'Month': 'date',
        'Latitude': 'latitude',
        'Longitude': 'longitude',
        'Crime type': 'crime_cat'
    })

    # Change month to date
    crime['date'] = pd.to_datetime(crime['date'], format='%Y-%m')

    # Drop duplicate rows
    dropped_rows['duplicates'] = crime.duplicated().sum()
    crime = crime.drop_duplicates()

    # Fill old crime ID with 'No ID' for crimes with no ID
    crime['old_crime_id'] = crime['old_crime_id'].fillna('No ID')

    # Remove rows with null locations
    dropped_rows['No location'] = crime.shape[0] - crime.dropna().shape[0]
    crime = crime.dropna()

    return crime, dropped_rows

In [5]:
dropped_rows = {}
clean_sth_yk, dropped_rows = Clean_Crime(raw_sth_yk, dropped_rows)

In [6]:
clean_sth_yk.head()

,old_crime_id,lsoa_code,date,latitude,longitude,crime_cat,Falls within
0,No ID,E01019456,2026-03-01,53.054017,-1.397019,Anti-social behaviour,South Yorkshire Police
1,No ID,E01007434,2026-03-01,53.599208,-1.451708,Anti-social behaviour,South Yorkshire Police
2,No ID,E01007434,2026-03-01,53.598502,-1.454483,Anti-social behaviour,South Yorkshire Police
3,No ID,E01007434,2026-03-01,53.603435,-1.446274,Anti-social behaviour,South Yorkshire Police
4,No ID,E01007434,2026-03-01,53.602548,-1.450879,Anti-social behaviour,South Yorkshire Police


In [7]:
for reason in dropped_rows:
    print(f'{reason}: {dropped_rows[reason]}')

duplicates: 658
No location: 668


#### Feature Engineering and Transformation

In [8]:
## Get PFA Crime falls within
clean_sth_yk['falls_within_stripped'] = clean_sth_yk['Falls within'].str.upper().str.strip('POLICE').str.replace(' ', '_')

clean_sth_yk.head()

,old_crime_id,lsoa_code,date,latitude,longitude,crime_cat,Falls within,falls_within_stripped
0,No ID,E01019456,2026-03-01,53.054017,-1.397019,Anti-social behaviour,South Yorkshire Police,SOUTH_YORKSHIRE_
1,No ID,E01007434,2026-03-01,53.599208,-1.451708,Anti-social behaviour,South Yorkshire Police,SOUTH_YORKSHIRE_
2,No ID,E01007434,2026-03-01,53.598502,-1.454483,Anti-social behaviour,South Yorkshire Police,SOUTH_YORKSHIRE_
3,No ID,E01007434,2026-03-01,53.603435,-1.446274,Anti-social behaviour,South Yorkshire Police,SOUTH_YORKSHIRE_
4,No ID,E01007434,2026-03-01,53.602548,-1.450879,Anti-social behaviour,South Yorkshire Police,SOUTH_YORKSHIRE_


In [9]:
## Get Year and Month of data
clean_sth_yk['date_str'] = clean_sth_yk['date'].dt.strftime('%Y%m')

clean_sth_yk.head()

,old_crime_id,lsoa_code,date,latitude,longitude,crime_cat,Falls within,falls_within_stripped,date_str
0,No ID,E01019456,2026-03-01,53.054017,-1.397019,Anti-social behaviour,South Yorkshire Police,SOUTH_YORKSHIRE_,202603
1,No ID,E01007434,2026-03-01,53.599208,-1.451708,Anti-social behaviour,South Yorkshire Police,SOUTH_YORKSHIRE_,202603
2,No ID,E01007434,2026-03-01,53.598502,-1.454483,Anti-social behaviour,South Yorkshire Police,SOUTH_YORKSHIRE_,202603
3,No ID,E01007434,2026-03-01,53.603435,-1.446274,Anti-social behaviour,South Yorkshire Police,SOUTH_YORKSHIRE_,202603
4,No ID,E01007434,2026-03-01,53.602548,-1.450879,Anti-social behaviour,South Yorkshire Police,SOUTH_YORKSHIRE_,202603


In [10]:
## Get 5 digit index of data
clean_sth_yk['index_id'] = (clean_sth_yk.index.astype(str).str.zfill(5))

clean_sth_yk.head()

,old_crime_id,lsoa_code,date,latitude,longitude,crime_cat,Falls within,falls_within_stripped,date_str,index_id
0,No ID,E01019456,2026-03-01,53.054017,-1.397019,Anti-social behaviour,South Yorkshire Police,SOUTH_YORKSHIRE_,202603,00000
1,No ID,E01007434,2026-03-01,53.599208,-1.451708,Anti-social behaviour,South Yorkshire Police,SOUTH_YORKSHIRE_,202603,00001
2,No ID,E01007434,2026-03-01,53.598502,-1.454483,Anti-social behaviour,South Yorkshire Police,SOUTH_YORKSHIRE_,202603,00002
3,No ID,E01007434,2026-03-01,53.603435,-1.446274,Anti-social behaviour,South Yorkshire Police,SOUTH_YORKSHIRE_,202603,00003
4,No ID,E01007434,2026-03-01,53.602548,-1.450879,Anti-social behaviour,South Yorkshire Police,SOUTH_YORKSHIRE_,202603,00004


In [11]:
# Finally, create a Crime ID using these 3 columns.
clean_sth_yk['crime_id'] = (clean_sth_yk['falls_within_stripped'] + clean_sth_yk['date_str'] + '_' + clean_sth_yk['index_id'])

clean_sth_yk.head()

,old_crime_id,lsoa_code,date,latitude,longitude,crime_cat,Falls within,falls_within_stripped,date_str,index_id,crime_id
0,No ID,E01019456,2026-03-01,53.054017,-1.397019,Anti-social behaviour,South Yorkshire Police,SOUTH_YORKSHIRE_,202603,00000,SOUTH_YORKSHIRE_202603_00000
1,No ID,E01007434,2026-03-01,53.599208,-1.451708,Anti-social behaviour,South Yorkshire Police,SOUTH_YORKSHIRE_,202603,00001,SOUTH_YORKSHIRE_202603_00001
2,No ID,E01007434,2026-03-01,53.598502,-1.454483,Anti-social behaviour,South Yorkshire Police,SOUTH_YORKSHIRE_,202603,00002,SOUTH_YORKSHIRE_202603_00002
3,No ID,E01007434,2026-03-01,53.603435,-1.446274,Anti-social behaviour,South Yorkshire Police,SOUTH_YORKSHIRE_,202603,00003,SOUTH_YORKSHIRE_202603_00003
4,No ID,E01007434,2026-03-01,53.602548,-1.450879,Anti-social behaviour,South Yorkshire Police,SOUTH_YORKSHIRE_,202603,00004,SOUTH_YORKSHIRE_202603_00004


In [12]:
clean_sth_yk.isnull().sum()

old_crime_id             0
lsoa_code                0
date                     0
latitude                 0
longitude                0
crime_cat                0
Falls within             0
falls_within_stripped    0
date_str                 0
index_id                 0
crime_id                 0
dtype: int64

In [13]:
clean_sth_yk.duplicated().sum()

np.int64(0)

Crime ID now created that will not replicate unless there are more than 100,000 entries per month per PFA.  
Function is as follows:

In [14]:
def Finalise_Crime(clean_crime_data):
    ## Get PFA Crime falls within
    clean_crime_data['falls_within_stripped'] = clean_crime_data['Falls within'].str.upper().str.strip('POLICE').str.replace(' ', '_')

    ## Get Year and Month of data
    clean_crime_data['date_str'] = clean_crime_data['date'].dt.strftime('%Y%m')

    ## Get 5 digit index of data
    clean_crime_data['index_id'] = (clean_crime_data.index.astype(str).str.zfill(5))

    # Finally, create a Crime ID using these 3 columns.
    clean_crime_data['crime_id'] = (clean_crime_data['falls_within_stripped'] + clean_crime_data['date_str'] + '_' + clean_crime_data['index_id'])

    # Clear columns and set an order
    final_crime_data = clean_crime_data[['crime_id', 'date', 'lsoa_code', 'latitude', 'longitude', 'crime_cat']]

    return final_crime_data

***
***
### Crime Data Aggregation

The end goal of this section is to create the general functions for importing all the data.

It will require a loop that names each individual police force, then each date for the data.  
within the loop, it should import that specific data.  
then clean the data, and check for issues with that.  
then create the same final database for the data, as is shown above.  
Finally, it should append the data to a large database holding all the data, and loop to the next entry.  

In [15]:
# Loop through each police force, then its dates

police_regions = ['merseyside', 'south-yorkshire', 'west-midlands', 'west-yorkshire']
years = ['2023', '2024', '2025', '2026']
months = ['01','02','03','04','05','06','07','08','09','10','11','12']

# Final DataFrame
aggregated_crime_data = pd.DataFrame(columns=['crime_id', 'date', 'lsoa_code', 'latitude', 'longitude', 'crime_cat'])

skipped_data = []
dropped_rows = {}
total_rows_dropped = 0

for police_region in police_regions:
    for year in years:
        for month in months:
            file_path = f'../Data/Raw/crime-data/{police_region}/{year}-{month}-{police_region}-street.csv'

            try:
                raw_crime_data = pd.read_csv(file_path)
                print(f'imported {police_region} {year} {month} data')
            except FileNotFoundError:
                print(f'Missing file: {file_path}, skipping...')
                skipped_data.append(f'{police_region}-{year}/{month}, reason: importing')
                continue

            clean_crime_data, dropped_rows = Clean_Crime(raw_crime_data, dropped_rows)

            print(f'Cleaned {police_region} {year} {month} data')
            total_rows_dropped += dropped_rows['No location'] # Add dropped rows with data to total_rows_dropped

            final_crime_data = Finalise_Crime(clean_crime_data)

            print(f'Finalised {police_region} {year} {month} data') # No rows dropped in this function

            # Check if clean
            if final_crime_data.duplicated().sum() != 0:
                # Drop duplicate rows and add them to the total rows dropped
                total_rows_dropped += final_crime_data.duplicated().sum()
                final_crime_data = final_crime_data.drop_duplicates()

                print(f'Duplicate values in final data, dropped')

            if final_crime_data.isnull().sum().sum() != 0:
                # Drop rows with nulls in
                total_rows_dropped += final_crime_data.shape[0] - final_crime_data.dropna().shape[0]
                final_crime_data = final_crime_data.dropna()

                print(f'Null values in final data, dropped')
            
            
            # Append to final data
            aggregated_crime_data = pd.concat([aggregated_crime_data, final_crime_data], ignore_index=True)
            print(f'Added {police_region} {year} {month} data to end data')

print(f'\nTotal rows in final Data: {aggregated_crime_data.shape[0]}')
print(f'Total rows dropped in cleaning and aggregation process: {total_rows_dropped}')
print(f'Percentage of data lost: {(total_rows_dropped/aggregated_crime_data.shape[0])*100}%')


Missing file: ../Data/Raw/crime-data/merseyside/2023-01-merseyside-street.csv, skipping...
Missing file: ../Data/Raw/crime-data/merseyside/2023-02-merseyside-street.csv, skipping...
Missing file: ../Data/Raw/crime-data/merseyside/2023-03-merseyside-street.csv, skipping...
imported merseyside 2023 04 data
Cleaned merseyside 2023 04 data
Finalised merseyside 2023 04 data


C:\Users\sam\AppData\Local\Temp\ipykernel_21680\2089479531.py:53: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  aggregated_crime_data = pd.concat([aggregated_crime_data, final_crime_data], ignore_index=True)


Added merseyside 2023 04 data to end data
imported merseyside 2023 05 data
Cleaned merseyside 2023 05 data
Finalised merseyside 2023 05 data
Added merseyside 2023 05 data to end data
imported merseyside 2023 06 data
Cleaned merseyside 2023 06 data
Finalised merseyside 2023 06 data
Added merseyside 2023 06 data to end data
imported merseyside 2023 07 data
Cleaned merseyside 2023 07 data
Finalised merseyside 2023 07 data
Added merseyside 2023 07 data to end data
imported merseyside 2023 08 data
Cleaned merseyside 2023 08 data
Finalised merseyside 2023 08 data
Added merseyside 2023 08 data to end data
imported merseyside 2023 09 data
Cleaned merseyside 2023 09 data
Finalised merseyside 2023 09 data
Added merseyside 2023 09 data to end data
imported merseyside 2023 10 data
Cleaned merseyside 2023 10 data
Finalised merseyside 2023 10 data
Added merseyside 2023 10 data to end data
imported merseyside 2023 11 data
Cleaned merseyside 2023 11 data
Finalised merseyside 2023 11 data
Added merseys

***
**All Crime data imported successfully**

Just over 1% of data lost in the process.
***

In [16]:
aggregated_crime_data.sample(10)

,crime_id,date,lsoa_code,latitude,longitude,crime_cat
26733,MERSEYSIDE_202305_13013,2023-05-01,E01007210,53.415641,-3.083751,Burglary
1728403,WEST_MIDLANDS_202507_28790,2025-07-01,E01010519,52.566246,-2.139623,Violence and sexual offences
1012021,WEST_MIDLANDS_202306_13292,2023-06-01,E01009200,52.476660,-1.866713,Other theft
1133397,WEST_MIDLANDS_202310_14357,2023-10-01,E01009570,52.428519,-1.498139,Violence and sexual offences
1342207,WEST_MIDLANDS_202405_29051,2024-05-01,E01010547,52.597196,-2.161689,Drugs
1775706,WEST_MIDLANDS_202509_20136,2025-09-01,E01010144,52.475193,-1.757668,Vehicle crime
1379996,WEST_MIDLANDS_202407_06760,2024-07-01,E01009363,52.467200,-1.872335,Criminal damage and arson
154858,MERSEYSIDE_202403_11040,2024-03-01,E01007122,53.400882,-3.056512,Vehicle crime
1295314,WEST_MIDLANDS_202404_09702,2024-04-01,E01009119,52.408063,-1.924427,Possession of weapons
238878,MERSEYSIDE_202410_07189,2024-10-01,E01033756,53.403105,-2.979105,Violence and sexual offences


#### Location Lookup

In [17]:
crime_location = pd.merge(aggregated_crime_data, location_lookup, how='left', on=['lsoa_code'])

crime_location.sample(10)

,crime_id,date,lsoa_code,latitude,longitude,crime_cat,lsoa_name,lad_code,lad_name,pfa_code,pfa_name
1152408,WEST_MIDLANDS_202311_02836,2023-11-01,E01009355,52.503039,-1.919506,Violence and sexual offences,Birmingham 039E,E08000025,Birmingham,E23000014,West Midlands
2774757,WEST_YORKSHIRE_202601_10292,2026-01-01,E01011227,53.626842,-1.786569,Vehicle crime,Kirklees 048C,E08000034,Kirklees,E23000010,West Yorkshire
1437241,WEST_MIDLANDS_202409_04592,2024-09-01,E01009346,52.490152,-1.942799,Possession of weapons,Birmingham 053E,E08000025,Birmingham,E23000014,West Midlands
2747813,WEST_YORKSHIRE_202512_05242,2025-12-01,E01033691,53.793840,-1.757443,Violence and sexual offences,Bradford 065B,E08000032,Bradford,E23000010,West Yorkshire
579531,SOUTH_YORKSHIRE_202401_00507,2024-01-01,E01007406,53.569571,-1.445417,Criminal damage and arson,Barnsley 009E,E08000038,Barnsley,E23000011,South Yorkshire
2628785,WEST_YORKSHIRE_202507_10562,2025-07-01,E01011130,53.689830,-1.631482,Violence and sexual offences,Kirklees 017B,E08000034,Kirklees,E23000010,West Yorkshire
1377807,WEST_MIDLANDS_202407_04516,2024-07-01,E01008913,52.494717,-1.899755,Violence and sexual offences,Birmingham 050B,E08000025,Birmingham,E23000014,West Midlands
2290473,WEST_YORKSHIRE_202405_22826,2024-05-01,E01033010,53.798896,-1.543208,Shoplifting,Leeds 111B,E08000035,Leeds,E23000010,West Yorkshire
1830245,WEST_MIDLANDS_202511_22045,2025-11-01,E01010383,52.599585,-2.035657,Public order,Walsall 019D,E08000030,Walsall,E23000014,West Midlands
2023817,WEST_YORKSHIRE_202307_05269,2023-07-01,E01010747,53.768363,-1.759334,Burglary,Bradford 056B,E08000032,Bradford,E23000010,West Yorkshire


In [18]:
crime_location.shape

(2830431, 11)

#### Population

In [19]:
# Import population data

population = pd.read_csv('../Data/Processed/population.csv')

In [20]:
crime_location['year'] = crime_location['date'].dt.year

In [21]:
crime_location_pop = pd.merge(crime_location, population, how='left', on=['lsoa_code', 'year'])

crime_location_pop.head()

,crime_id,date,lsoa_code,latitude,longitude,crime_cat,lsoa_name,lad_code,lad_name,pfa_code,pfa_name,year,population
0,MERSEYSIDE_202304_00000,2023-04-01,E01018537,53.301368,-3.089063,Anti-social behaviour,Cheshire West and Chester 001D,E06000050,Cheshire West and Chester,E23000006,Cheshire,2023,2018.0
1,MERSEYSIDE_202304_00001,2023-04-01,E01018537,53.315654,-3.074956,Vehicle crime,Cheshire West and Chester 001D,E06000050,Cheshire West and Chester,E23000006,Cheshire,2023,2018.0
2,MERSEYSIDE_202304_00002,2023-04-01,E01018570,53.300746,-2.960659,Other crime,Cheshire West and Chester 004C,E06000050,Cheshire West and Chester,E23000006,Cheshire,2023,1619.0
3,MERSEYSIDE_202304_00003,2023-04-01,E01012393,53.389101,-2.746819,Violence and sexual offences,Halton 001B,E06000006,Halton,E23000006,Cheshire,2023,2948.0
4,MERSEYSIDE_202304_00004,2023-04-01,E01012376,53.377272,-2.756590,Anti-social behaviour,Halton 002C,E06000006,Halton,E23000006,Cheshire,2023,2822.0


In [22]:
crime_location_pop.shape

(2830431, 13)

#### Deprivation

In [23]:
# Import Deprivation

deprivation = pd.read_csv('../Data/Processed/deprivation.csv')

In [24]:
crime_loc_pop_depr = pd.merge(crime_location_pop, deprivation, how='left', on='lsoa_code')

crime_loc_pop_depr.sample(10)

,crime_id,date,lsoa_code,latitude,longitude,crime_cat,lsoa_name,lad_code,lad_name,pfa_code,pfa_name,year,population,imd_score,incm_score,empl_score,edcn_score,hous_score
337091,MERSEYSIDE_202506_09144,2025-06-01,E01007009,53.444475,-2.993109,Violence and sexual offences,Sefton 037D,E08000014,Sefton,E23000004,Merseyside,2025,2196.0,62.119,0.571,0.363,63.621,10.612
1992405,WEST_YORKSHIRE_202306_01841,2023-06-01,E01010605,53.822057,-1.737866,Theft from the person,Bradford 024C,E08000032,Bradford,E23000010,West Yorkshire,2023,1601.0,47.490,0.467,0.257,69.531,11.990
591975,SOUTH_YORKSHIRE_202401_13390,2024-01-01,E01033265,53.380913,-1.475847,Violence and sexual offences,Sheffield 074C,E08000039,Sheffield,E23000011,South Yorkshire,2024,2077.0,24.739,0.124,0.074,10.105,22.351
1594404,WEST_MIDLANDS_202503_04496,2025-03-01,E01009151,52.483916,-1.948432,Violence and sexual offences,Birmingham 053A,E08000025,Birmingham,E23000014,West Midlands,2025,2279.5,51.261,0.570,0.255,55.422,24.624
47011,MERSEYSIDE_202307_03232,2023-07-01,E01006646,53.414840,-2.970993,Anti-social behaviour,Liverpool 023C,E08000012,Liverpool,E23000004,Merseyside,2023,2330.0,53.765,0.549,0.291,46.992,14.338
1109581,WEST_MIDLANDS_202309_19776,2023-09-01,E01010093,52.529837,-2.014771,Other theft,Sandwell 013A,E08000028,Sandwell,E23000014,West Midlands,2023,1684.0,34.187,0.370,0.209,38.194,24.963
752468,SOUTH_YORKSHIRE_202501_09979,2025-01-01,E01007957,53.373941,-1.551793,Other crime,Sheffield 041C,E08000039,Sheffield,E23000011,South Yorkshire,2025,1620.0,2.549,0.039,0.041,0.301,19.416
1083462,WEST_MIDLANDS_202308_22998,2023-08-01,E01010109,52.465682,-1.737054,Robbery,Solihull 009A,E08000029,Solihull,E23000014,West Midlands,2023,1709.0,18.434,0.178,0.119,19.117,23.559
263628,MERSEYSIDE_202412_06753,2024-12-01,E01007057,53.652138,-2.963760,Other theft,Sefton 005C,E08000014,Sefton,E23000004,Merseyside,2024,1531.0,23.755,0.274,0.166,24.828,13.463
1588510,WEST_MIDLANDS_202502_23104,2025-02-01,E01010564,52.596436,-2.090720,Violence and sexual offences,Wolverhampton 012C,E08000031,Wolverhampton,E23000014,West Midlands,2025,2901.5,30.521,0.330,0.186,28.150,21.464


In [25]:
crime_loc_pop_depr.shape

(2830431, 18)

#### Crime Severity

In [26]:
# Import Crime severity data

severity = pd.read_csv('../Data/Processed/crime-severity.csv')

In [27]:
severity.head()
severity = severity.drop(columns='Unnamed: 0')

In [28]:
# Modify crime cat to be lower case
crime_loc_pop_depr['crime_cat'] = crime_loc_pop_depr['crime_cat'].str.lower()

In [29]:
crime_loc_pop_depr.head()

,crime_id,date,lsoa_code,latitude,longitude,crime_cat,lsoa_name,lad_code,lad_name,pfa_code,pfa_name,year,population,imd_score,incm_score,empl_score,edcn_score,hous_score
0,MERSEYSIDE_202304_00000,2023-04-01,E01018537,53.301368,-3.089063,anti-social behaviour,Cheshire West and Chester 001D,E06000050,Cheshire West and Chester,E23000006,Cheshire,2023,2018.0,4.137,0.050,0.049,0.889,18.163
1,MERSEYSIDE_202304_00001,2023-04-01,E01018537,53.315654,-3.074956,vehicle crime,Cheshire West and Chester 001D,E06000050,Cheshire West and Chester,E23000006,Cheshire,2023,2018.0,4.137,0.050,0.049,0.889,18.163
2,MERSEYSIDE_202304_00002,2023-04-01,E01018570,53.300746,-2.960659,other crime,Cheshire West and Chester 004C,E06000050,Cheshire West and Chester,E23000006,Cheshire,2023,1619.0,10.524,0.087,0.060,4.840,26.361
3,MERSEYSIDE_202304_00003,2023-04-01,E01012393,53.389101,-2.746819,violence and sexual offences,Halton 001B,E06000006,Halton,E23000006,Cheshire,2023,2948.0,4.483,0.051,0.051,2.076,16.072
4,MERSEYSIDE_202304_00004,2023-04-01,E01012376,53.377272,-2.756590,anti-social behaviour,Halton 002C,E06000006,Halton,E23000006,Cheshire,2023,2822.0,5.531,0.053,0.053,3.032,21.859


In [30]:
crime_loc_pop_depr_sev = pd.merge(crime_loc_pop_depr, severity, how='left', on='crime_cat')

crime_loc_pop_depr_sev.sample(10)

,crime_id,date,lsoa_code,latitude,longitude,crime_cat,lsoa_name,lad_code,lad_name,pfa_code,pfa_name,year,population,imd_score,incm_score,empl_score,edcn_score,hous_score,avg_weight
271561,MERSEYSIDE_202501_03354,2025-01-01,E01006695,53.415813,-2.944986,public order,Liverpool 028C,E08000012,Liverpool,E23000004,Merseyside,2025,2251.5,73.213,0.604,0.430,42.821,20.713,365.0
1708263,WEST_MIDLANDS_202507_08350,2025-07-01,E01009187,52.443552,-1.886218,other theft,Birmingham 092C,E08000025,Birmingham,E23000014,West Midlands,2025,1469.0,16.958,0.160,0.145,3.250,13.230,51.5
1984123,WEST_YORKSHIRE_202305_23835,2023-05-01,E01011491,53.750204,-1.558855,possession of weapons,Leeds 101A,E08000035,Leeds,E23000010,West Yorkshire,2023,1477.0,57.378,0.549,0.316,57.942,24.765,75.0
1839314,WEST_MIDLANDS_202512_05346,2025-12-01,E01009329,52.467685,-1.856539,robbery,Birmingham 070A,E08000025,Birmingham,E23000014,West Midlands,2025,2910.0,58.739,0.702,0.269,47.017,30.261,746.0
450222,MERSEYSIDE_202603_11851,2026-03-01,E01007158,53.394932,-3.057538,criminal damage and arson,Wirral 015D,E08000015,Wirral,E23000004,Merseyside,2026,1517.0,14.379,0.126,0.123,9.085,3.202,19.0
311999,MERSEYSIDE_202504_09423,2025-04-01,E01006845,53.441862,-2.713431,violence and sexual offences,St. Helens 014A,E08000013,St. Helens,E23000004,Merseyside,2025,2482.0,31.718,0.321,0.196,35.386,12.408,709.0
2093413,WEST_YORKSHIRE_202309_21005,2023-09-01,E01011307,53.750912,-1.378096,vehicle crime,Leeds 103A,E08000035,Leeds,E23000010,West Yorkshire,2023,1599.0,26.115,0.220,0.164,34.214,25.387,41.0
1995309,WEST_YORKSHIRE_202306_04848,2023-06-01,E01033697,53.786070,-1.788495,public order,Bradford 049J,E08000032,Bradford,E23000010,West Yorkshire,2023,2911.0,57.474,0.570,0.283,61.659,22.450,365.0
1890706,WEST_MIDLANDS_202602_08279,2026-02-01,E01008978,52.404744,-1.898444,violence and sexual offences,Birmingham 121B,E08000025,Birmingham,E23000014,West Midlands,2026,1051.0,77.901,0.630,0.491,72.070,25.249,709.0
2217950,WEST_YORKSHIRE_202402_21820,2024-02-01,E01011933,53.636760,-1.527159,anti-social behaviour,Wakefield 036C,E08000036,Wakefield,E23000010,West Yorkshire,2024,1628.0,23.977,0.209,0.143,32.100,24.707,NaN


In [31]:
crime_loc_pop_depr_sev.shape

(2830431, 19)

***
Only Anti-social behaviour is null crime category.  
The crime severity weighting is based off of the severity of court rulings coming from crimes. As ASB often goes unpunished, it does not have a crime severity weighting score.  
**assumption:** Crime severity for ASB is 1. This is a low enough number to barely weigh in to crime severity, but does not leave it unnoticed.
***

In [32]:
crime_loc_pop_depr_sev['avg_weight'] = crime_loc_pop_depr_sev['avg_weight'].fillna(1)

In [33]:
crime_loc_pop_depr_sev.head()

,crime_id,date,lsoa_code,latitude,longitude,crime_cat,lsoa_name,lad_code,lad_name,pfa_code,pfa_name,year,population,imd_score,incm_score,empl_score,edcn_score,hous_score,avg_weight
0,MERSEYSIDE_202304_00000,2023-04-01,E01018537,53.301368,-3.089063,anti-social behaviour,Cheshire West and Chester 001D,E06000050,Cheshire West and Chester,E23000006,Cheshire,2023,2018.0,4.137,0.050,0.049,0.889,18.163,1.0
1,MERSEYSIDE_202304_00001,2023-04-01,E01018537,53.315654,-3.074956,vehicle crime,Cheshire West and Chester 001D,E06000050,Cheshire West and Chester,E23000006,Cheshire,2023,2018.0,4.137,0.050,0.049,0.889,18.163,41.0
2,MERSEYSIDE_202304_00002,2023-04-01,E01018570,53.300746,-2.960659,other crime,Cheshire West and Chester 004C,E06000050,Cheshire West and Chester,E23000006,Cheshire,2023,1619.0,10.524,0.087,0.060,4.840,26.361,86.0
3,MERSEYSIDE_202304_00003,2023-04-01,E01012393,53.389101,-2.746819,violence and sexual offences,Halton 001B,E06000006,Halton,E23000006,Cheshire,2023,2948.0,4.483,0.051,0.051,2.076,16.072,709.0
4,MERSEYSIDE_202304_00004,2023-04-01,E01012376,53.377272,-2.756590,anti-social behaviour,Halton 002C,E06000006,Halton,E23000006,Cheshire,2023,2822.0,5.531,0.053,0.053,3.032,21.859,1.0


In [34]:
crime_final = crime_loc_pop_depr_sev[[
    'crime_id', 
    'date', 
    'year', 
    'pfa_code', 
    'pfa_name', 
    'latitude', 
    'longitude', 
    'lsoa_code', 
    'lsoa_name',
    'lad_code',
    'lad_name',
    'crime_cat',
    'avg_weight',
    'population',
    'imd_score',
    'incm_score',
    'empl_score',
    'edcn_score',
    'hous_score'
]]

crime_final.head()

,crime_id,date,year,pfa_code,pfa_name,latitude,longitude,lsoa_code,lsoa_name,lad_code,lad_name,crime_cat,avg_weight,population,imd_score,incm_score,empl_score,edcn_score,hous_score
0,MERSEYSIDE_202304_00000,2023-04-01,2023,E23000006,Cheshire,53.301368,-3.089063,E01018537,Cheshire West and Chester 001D,E06000050,Cheshire West and Chester,anti-social behaviour,1.0,2018.0,4.137,0.050,0.049,0.889,18.163
1,MERSEYSIDE_202304_00001,2023-04-01,2023,E23000006,Cheshire,53.315654,-3.074956,E01018537,Cheshire West and Chester 001D,E06000050,Cheshire West and Chester,vehicle crime,41.0,2018.0,4.137,0.050,0.049,0.889,18.163
2,MERSEYSIDE_202304_00002,2023-04-01,2023,E23000006,Cheshire,53.300746,-2.960659,E01018570,Cheshire West and Chester 004C,E06000050,Cheshire West and Chester,other crime,86.0,1619.0,10.524,0.087,0.060,4.840,26.361
3,MERSEYSIDE_202304_00003,2023-04-01,2023,E23000006,Cheshire,53.389101,-2.746819,E01012393,Halton 001B,E06000006,Halton,violence and sexual offences,709.0,2948.0,4.483,0.051,0.051,2.076,16.072
4,MERSEYSIDE_202304_00004,2023-04-01,2023,E23000006,Cheshire,53.377272,-2.756590,E01012376,Halton 002C,E06000006,Halton,anti-social behaviour,1.0,2822.0,5.531,0.053,0.053,3.032,21.859


***
***
### Final Database Check and Export

In [35]:
# Check cleanliness
crime_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2830431 entries, 0 to 2830430
Data columns (total 19 columns):
 #   Column      Dtype         
---  ------      -----         
 0   crime_id    object        
 1   date        datetime64[ns]
 2   year        int32         
 3   pfa_code    object        
 4   pfa_name    object        
 5   latitude    float64       
 6   longitude   float64       
 7   lsoa_code   object        
 8   lsoa_name   object        
 9   lad_code    object        
 10  lad_name    object        
 11  crime_cat   object        
 12  avg_weight  float64       
 13  population  float64       
 14  imd_score   float64       
 15  incm_score  float64       
 16  empl_score  float64       
 17  edcn_score  float64       
 18  hous_score  float64       
dtypes: datetime64[ns](1), float64(9), int32(1), object(8)
memory usage: 399.5+ MB


In [36]:
crime_final.isnull().sum()

crime_id         0
date             0
year             0
pfa_code      7569
pfa_name      7569
latitude         0
longitude        0
lsoa_code        0
lsoa_name     7569
lad_code      7569
lad_name      7569
crime_cat        0
avg_weight       0
population    7569
imd_score     7573
incm_score    7573
empl_score    7573
edcn_score    7573
hous_score    7573
dtype: int64

In [37]:
crime_final[crime_final['pfa_code'].isnull()]['lsoa_code'].value_counts()

lsoa_code
E01007645    469
E01009642    450
E01010521    369
E01010994    346
E01011916    292
            ... 
E01007859      5
E01033766      4
E01033749      3
E01024618      1
E01024929      1
Name: count, Length: 104, dtype: int64

***
For some reason the location lookup table seems to be missing some lsoa codes - likely due to a mismatch in the year of intake.  
Unfortunately it is too late in the project to fix this, so the rows will have to be dropped.
***

In [38]:
total_rows_dropped += crime_final.shape[0] - crime_final.dropna().shape[0]
crime_final = crime_final.dropna()

print(f'\nTotal rows in final Data: {crime_final.shape[0]}')
print(f'Total rows dropped in cleaning and aggregation process: {total_rows_dropped}')
print(f'Percentage of data lost: {(total_rows_dropped/crime_final.shape[0])*100}%')


Total rows in final Data: 2822858
Total rows dropped in cleaning and aggregation process: 41197
Percentage of data lost: 1.4594074515969275%


In [39]:
crime_final.duplicated().sum()

np.int64(0)

In [40]:
crime_final = crime_final.rename(columns={'avg_weight':'crime_sev'})

***
**Final data is now fully cclean and ready to export**
***

In [41]:
# Final check
crime_final.sample(10)

,crime_id,date,year,pfa_code,pfa_name,latitude,longitude,lsoa_code,lsoa_name,lad_code,lad_name,crime_cat,crime_sev,population,imd_score,incm_score,empl_score,edcn_score,hous_score
2499793,WEST_YORKSHIRE_202502_04721,2025-02-01,2025,E23000010,West Yorkshire,53.744903,-1.768035,E01010869,Bradford 061B,E08000032,Bradford,vehicle crime,41.0,1699.0,33.560,0.307,0.191,36.745,16.774
1939301,WEST_YORKSHIRE_202304_06299,2023-04-01,2023,E23000010,West Yorkshire,53.761447,-1.684107,E01010816,Bradford 057C,E08000032,Bradford,public order,365.0,1948.0,13.258,0.123,0.093,17.645,21.629
1303379,WEST_MIDLANDS_202404_17929,2024-04-01,2024,E23000014,West Midlands,52.457466,-2.144545,E01009856,Dudley 035A,E08000027,Dudley,shoplifting,13.0,2054.0,26.231,0.297,0.166,22.274,14.595
392457,MERSEYSIDE_202510_13406,2025-10-01,2025,E23000004,Merseyside,53.343128,-3.108962,E01007288,Wirral 037E,E08000015,Wirral,violence and sexual offences,709.0,1749.0,17.747,0.261,0.140,14.068,17.702
1305549,WEST_MIDLANDS_202404_20136,2024-04-01,2024,E23000014,West Midlands,52.496538,-1.982811,E01010050,Sandwell 023E,E08000028,Sandwell,violence and sexual offences,709.0,1855.0,27.384,0.326,0.148,41.823,23.556
565856,SOUTH_YORKSHIRE_202311_12731,2023-11-01,2023,E23000011,South Yorkshire,53.388087,-1.473940,E01033262,Sheffield 073B,E08000039,Sheffield,vehicle crime,41.0,2659.0,14.124,0.141,0.053,2.905,18.937
29098,MERSEYSIDE_202305_15465,2023-05-01,2023,E23000004,Merseyside,53.328614,-2.982487,E01007136,Wirral 039B,E08000015,Wirral,violence and sexual offences,709.0,1648.0,35.776,0.396,0.254,33.352,6.821
43326,MERSEYSIDE_202306_14466,2023-06-01,2023,E23000004,Merseyside,53.374543,-3.163922,E01007260,Wirral 028B,E08000015,Wirral,other theft,51.5,1560.0,4.577,0.064,0.077,2.525,10.307
1354276,WEST_MIDLANDS_202406_10348,2024-06-01,2024,E23000014,West Midlands,52.407715,-1.928190,E01009119,Birmingham 123C,E08000025,Birmingham,burglary,438.0,1398.0,21.802,0.212,0.161,17.059,11.153
1246596,WEST_MIDLANDS_202402_14654,2024-02-01,2024,E23000014,West Midlands,52.388855,-1.545080,E01033058,Coventry 042F,E08000026,Coventry,other theft,51.5,3388.0,14.573,0.145,0.075,24.618,20.148


In [42]:
# Export to csv file

## Output Final Table to csv
crime_final.to_csv('../Data/Output/crime-final.csv', index=True)

print(f'crime_final.csv File successfully created: {Path('../Data/Output/crime-final.csv').exists()}')

crime_final.csv File successfully created: True
